In [13]:
import pandas as pd 
import numpy as np 
import os 

In [14]:
metadata = pd.read_csv('../../data/metadata_demographic.csv')
metadata = metadata.drop_duplicates(subset='Participant_ID')
metadata = metadata.dropna(subset=['pd'])
metadata

,Protocol,Participant_ID,Task,gender,age,race,pd
0,SuperPD,NIHFT628PHTAY,ahhhh,female,70.0,White,no
1,SuperPD,NIHNT179KNNF4,ahhhh,female,70.0,White,yes
2,SuperPD,NIHYM875FLXFF,ahhhh,female,73.0,White,no
3,SuperPD,NIHBV117HUCTC,ahhhh,female,60.0,White,no
4,SuperPD,NIHZY217YWJA8,ahhhh,female,69.0,White,yes
...,...,...,...,...,...,...,...
1996,ValorPD,MOyJjyLX9hPvP3FSlLNYaG28xA23,NaN,M,78.0,White,yes
1997,ValorPD,zn6iI7uiq0U0xWTG5fuwR7yX9IH3,NaN,F,62.0,White,no
1998,ValorPD,XFJkSNMEpNgUnWg5Ouc6AWMOnQ82,NaN,F,62.0,White,yes
1999,ValorPD,j4GZI8ZFugZHRPxCKLBHC3DFwZE2,NaN,M,70.0,White,yes


In [15]:
BASE_DIR_DEMO_MATCHED = '../../../UFNet_Carrier_Classification_Demo_Matched/'

with open(os.path.join(BASE_DIR_DEMO_MATCHED,"data/train_set_participants_demo_matched.txt")) as f:
    ids = f.readlines()
    train_ids = set([x.strip() for x in ids])

with open(os.path.join(BASE_DIR_DEMO_MATCHED,"data/dev_set_participants_demo_matched.txt")) as f:
    ids = f.readlines()
    dev_ids = set([x.strip() for x in ids])

with open(os.path.join(BASE_DIR_DEMO_MATCHED,"data/test_set_participants_demo_matched.txt")) as f:
    ids = f.readlines()
    test_ids = set([x.strip() for x in ids])

all_ids = train_ids.union(dev_ids)
all_ids = all_ids.union(test_ids)

print(len(all_ids))

305


In [16]:
metadata = metadata[metadata['Participant_ID'].isin(all_ids)]
metadata.shape

(305, 7)

In [17]:
def create_distribution_table_with_order(df, diagnosis_col, target_col, first_row_text, value_order=None):
    """
    Create a table showing counts and percentages for a specified column split by a diagnosis column.
    
    Parameters:
        df (pd.DataFrame): The input dataset.
        diagnosis_col (str): The column that indicates diagnosis (e.g., "Diagnosis").
        target_col (str): The column for which distribution is calculated (e.g., "Sex").
        first_row_text (str): The first row text for the table header (e.g., "Sex, n (%)").
        value_order (list): The preferred order of column values (e.g., ['Male', 'Female', ...]).
        
    Returns:
        pd.DataFrame: The distribution table.
    """
    # Calculate counts within each group (With PD and Without PD)
    with_pd_total = df[df[diagnosis_col] == 1].shape[0]
    without_pd_total = df[df[diagnosis_col] != 1].shape[0]

    # Counts for each target value in the groups
    with_pd_counts = df[df[diagnosis_col] == 1][target_col].value_counts()
    without_pd_counts = df[df[diagnosis_col] != 1][target_col].value_counts()
    total_counts = df[target_col].value_counts()

    # Use value_order or infer from the data
    if value_order is None:
        value_order = total_counts.index.tolist()

    # Create the rows for each unique value in the preferred order
    rows = []
    for value in value_order:
        # Get counts for each group
        with_pd_count = with_pd_counts.get(value, 0)
        without_pd_count = without_pd_counts.get(value, 0)
        total_count = total_counts.get(value, 0)

        # Calculate percentages within each group
        with_pd_pct = (with_pd_count / with_pd_total) * 100 if with_pd_total > 0 else 0
        without_pd_pct = (without_pd_count / without_pd_total) * 100 if without_pd_total > 0 else 0
        total_pct = (total_count / df.shape[0]) * 100

        # Add row to the table
        rows.append([
            first_row_text,
            value,
            f"{with_pd_count} ({with_pd_pct:.1f}%)",
            f"{without_pd_count} ({without_pd_pct:.1f}%)",
            f"{total_count} ({total_pct:.1f}%)"
        ])

    # Create the table DataFrame
    table = pd.DataFrame(rows, columns=["Demographic Property", "Attribute", "LRRK2 Carrier", "Control", "Total"])

    # # Add the first row text
    # first_row = pd.DataFrame([{
    #     "Demographic Property": first_row_text,
    #     "Attribute": "",
    #     "With PD": "",
    #     "Without PD": "",
    #     "Total": ""
    # }])

    # # Concatenate the header and the data rows
    # table = pd.concat([first_row, table], ignore_index=True)

    for index in range(1, len(table)):
        table.loc[index, 'Demographic Property'] = ''
    
    return table


In [18]:
metadata['Protocol'].value_counts(dropna=False)

Protocol
ParkTest           139
ValorPD            122
ValidationStudy     19
SuperPD_old         11
ClusterPD            7
SuperPD              5
InMotion             2
Name: count, dtype: int64

In [19]:
metadata['carrier'] = 0
# take all Protocol = ValorPD, and set carrier = 1
metadata.loc[metadata['Protocol'] == 'ValorPD', 'carrier'] = 1

/tmp/ipykernel_2543543/2910501634.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['carrier'] = 0


In [20]:
metadata['carrier'].value_counts(dropna=False)

carrier
0    183
1    122
Name: count, dtype: int64

In [21]:
# Calculate counts and percentages for 'With PD' and 'Without PD'
with_pd_count = (metadata['carrier'] == 1).sum()  # Assuming '1' indicates 'With PD'
without_pd_count = (metadata['carrier'] != 1).sum()
total_count = with_pd_count + without_pd_count

with_pd_percentage = (with_pd_count / total_count) * 100
without_pd_percentage = (without_pd_count / total_count) * 100

# Create the final table structure
table_dict = {
    "Demographic Property": "",
    "Attribute": ["Number of Participants, n (%)"],
    "LRRK2 Carrier": [f"{with_pd_count} ({with_pd_percentage:.1f}%)"],
    "Control": [f"{without_pd_count} ({without_pd_percentage:.1f}%)"],
    "Total": [f"{total_count} (100%)"]
}

# Convert to a DataFrame for display
pd_count_table = pd.DataFrame(table_dict)

pd_count_table

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,,"Number of Participants, n (%)",122 (40.0%),183 (60.0%),305 (100%)


In [22]:
metadata['gender'].value_counts()

gender
female    95
F         71
male      69
M         51
Female    10
Male       9
Name: count, dtype: int64

In [23]:
gender_map = {
    "female": "Female",
    "Female": "Female",
    'F': 'Female',
    "male": "Male",
    "Male": "Male",
    'M': 'Male',
    "Prefer not to respond": "Unknown",
    "Nonbinary": "Non-Binary"
}
metadata['gender_normalized'] = metadata['gender'].map(gender_map)
metadata['gender_normalized'].value_counts(dropna=False)

/tmp/ipykernel_2543543/2488056917.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['gender_normalized'] = metadata['gender'].map(gender_map)


gender_normalized
Female    176
Male      129
Name: count, dtype: int64

In [24]:
# Define the preferred order
preferred_order = ['Female', 'Male',]

# Call the function with the preferred order
sex_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="carrier",
    target_col="gender_normalized",
    first_row_text="Sex",
    value_order=preferred_order
)


sex_table_with_order

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,Sex,Female,71 (58.2%),105 (57.4%),176 (57.7%)
1,,Male,51 (41.8%),78 (42.6%),129 (42.3%)


In [25]:
metadata['age'].value_counts(dropna=False)

age
66.0    19
60.0    16
57.0    14
62.0    13
59.0    12
68.0    12
65.0    11
64.0    10
58.0    10
63.0     9
56.0     9
55.0     8
61.0     8
34.0     7
76.0     7
67.0     7
69.0     7
NaN      7
53.0     6
51.0     6
70.0     6
29.0     5
52.0     5
73.0     5
27.0     5
32.0     5
48.0     5
37.0     4
50.0     4
71.0     4
42.0     4
25.0     4
30.0     4
31.0     3
40.0     3
36.0     3
39.0     3
43.0     3
72.0     3
23.0     3
24.0     3
38.0     3
75.0     2
74.0     2
35.0     2
41.0     2
49.0     2
44.0     1
77.0     1
22.0     1
54.0     1
21.0     1
26.0     1
87.0     1
33.0     1
79.0     1
45.0     1
Name: count, dtype: int64

In [26]:
metadata['age'] = metadata['age'].apply(lambda x: np.nan if x < 15 or x > 100 else x)
# Display the value counts of the 'age' column sorted by ascending order of the age values
metadata['age'].value_counts(dropna=False).sort_index()


/tmp/ipykernel_2543543/2598866290.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age'] = metadata['age'].apply(lambda x: np.nan if x < 15 or x > 100 else x)


age
21.0     1
22.0     1
23.0     3
24.0     3
25.0     4
26.0     1
27.0     5
29.0     5
30.0     4
31.0     3
32.0     5
33.0     1
34.0     7
35.0     2
36.0     3
37.0     4
38.0     3
39.0     3
40.0     3
41.0     2
42.0     4
43.0     3
44.0     1
45.0     1
48.0     5
49.0     2
50.0     4
51.0     6
52.0     5
53.0     6
54.0     1
55.0     8
56.0     9
57.0    14
58.0    10
59.0    12
60.0    16
61.0     8
62.0    13
63.0     9
64.0    10
65.0    11
66.0    19
67.0     7
68.0    12
69.0     7
70.0     6
71.0     4
72.0     3
73.0     5
74.0     2
75.0     2
76.0     7
77.0     1
79.0     1
87.0     1
NaN      7
Name: count, dtype: int64

In [27]:
# Initialize 'age_normalized' column with NaN
metadata['age_normalized'] = np.nan

# Ensure 'age' is numeric where possible
def safe_numeric(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return np.nan

metadata['age_numeric'] = metadata['age'].apply(safe_numeric)

# Define conditions for numeric age ranges
conditions = [
    metadata['age_numeric'] < 20,
    (metadata['age_numeric'] >= 20) & (metadata['age_numeric'] <= 39),
    (metadata['age_numeric'] >= 40) & (metadata['age_numeric'] <= 59),
    (metadata['age_numeric'] >= 60) & (metadata['age_numeric'] <= 79),
    metadata['age_numeric'] >= 80
]

# Define labels for the conditions
age_labels = [
    '< 20',
    '20 - 39',
    '40 - 59',
    '60 - 79',
    '>= 80'
]

# Apply conditions to normalize 'age'
metadata['age_normalized'] = np.select(
    conditions,
    age_labels,
    default='Not Mentioned'
)


/tmp/ipykernel_2543543/3161752875.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age_normalized'] = np.nan
/tmp/ipykernel_2543543/3161752875.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age_numeric'] = metadata['age'].apply(safe_numeric)
/tmp/ipykernel_2543543/3161752875.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://panda

In [28]:
import numpy as np

def process_age(age):
    try:
        # If the age is already numeric, return it
        if isinstance(age, (int, float)):
            return float(age)
        # If the age is a range like "50-60", calculate the mean
        if isinstance(age, str) and '-' in age:
            start, end = map(float, age.split('-'))
            return (start + end) / 2
    except:
        pass
    # Return NaN for invalid entries
    return np.nan

# Apply the processing function to handle all cases
metadata['age_processed'] = metadata['age'].apply(process_age)

# Calculate the mean and range (ignoring NaN values)
mean_age = metadata['age_processed'].mean()
min_age = metadata['age_processed'].min()
max_age = metadata['age_processed'].max()

# Print the results
print(f"Mean age: {mean_age:.2f}")
print(f"Age range: {min_age:.2f} - {max_age:.2f}")


Mean age: 55.17
Age range: 21.00 - 87.00


/tmp/ipykernel_2543543/563915863.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['age_processed'] = metadata['age'].apply(process_age)


In [29]:
metadata['age_normalized'].value_counts(dropna=False)

age_normalized
60 - 79          143
40 - 59           96
20 - 39           58
Not Mentioned      7
>= 80              1
Name: count, dtype: int64

In [30]:
# Define the preferred order
preferred_order = [
    '< 20',
    '20 - 39',
    '40 - 59',
    '60 - 79',
    '>= 80',
    'Not Mentioned'
]



# Call the function with the preferred order
age_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="carrier",
    target_col="age_normalized",
    first_row_text=f"Age in years (range: {min_age:.1f} - {max_age:.1f}, mean: {mean_age:.1f}), n (%)",
    value_order=preferred_order
)


age_table_with_order

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,"Age in years (range: 21.0 - 87.0, mean: 55.2),...",< 20,0 (0.0%),0 (0.0%),0 (0.0%)
1,,20 - 39,35 (28.7%),23 (12.6%),58 (19.0%)
2,,40 - 59,38 (31.1%),58 (31.7%),96 (31.5%)
3,,60 - 79,49 (40.2%),94 (51.4%),143 (46.9%)
4,,>= 80,0 (0.0%),1 (0.5%),1 (0.3%)
5,,Not Mentioned,0 (0.0%),7 (3.8%),7 (2.3%)


In [31]:
metadata['race'].value_counts(dropna=False)

race
White                   139
white,                  108
white,race               27
['White']                18
Prefer Not to Answer      6
NaN                       2
['Asian', 'White']        1
white                     1
white,black,              1
asian,white,              1
asian,race                1
Name: count, dtype: int64

In [32]:
def map_race_simplified(value):
    if isinstance(value, str):
        value = value.lower()  # Make case-insensitive
        if 'asian' in value:
            return "Asian"
        elif 'nativeamerican' in value or 'american indian' in value:
            return "American Indian or Alaska Native"
        elif 'black' in value:
            return "Black or African American"
        elif 'nativepacific' in value or 'hawaiian' in value:
            return "Native Hawaiian or Other Pacific Islander"
        elif 'white' in value:
            return "White"
        elif 'other' in value or 'on' in value:
            return "Others"
        elif 'prefer not to' in value or '[]' in value:
            return "Not Mentioned"
    return "Not Mentioned"  # Default to Unknown if value is not a string or doesn't match

# Apply the function to the 'race' column
metadata['race_normalized'] = metadata['race'].apply(map_race_simplified)

# Display the cleaned race column counts
print(metadata['race_normalized'].value_counts(dropna=False))

race_normalized
White                        293
Not Mentioned                  8
Asian                          3
Black or African American      1
Name: count, dtype: int64


/tmp/ipykernel_2543543/2427306195.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['race_normalized'] = metadata['race'].apply(map_race_simplified)


In [33]:
# Define the preferred order
preferred_order = [
    "White",
    "Asian",
    "Black or African American",
    "American Indian or Alaska Native",
    "Others",
    "Not Mentioned"
]

# Call the function with the preferred order
race_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="carrier",
    target_col="race_normalized",
    first_row_text="Race, n (%)",
    value_order=preferred_order
)


race_table_with_order

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,"Race, n (%)",White,116 (95.1%),177 (96.7%),293 (96.1%)
1,,Asian,0 (0.0%),3 (1.6%),3 (1.0%)
2,,Black or African American,0 (0.0%),1 (0.5%),1 (0.3%)
3,,American Indian or Alaska Native,0 (0.0%),0 (0.0%),0 (0.0%)
4,,Others,0 (0.0%),0 (0.0%),0 (0.0%)
5,,Not Mentioned,6 (4.9%),2 (1.1%),8 (2.6%)


In [34]:
metadata['Protocol'].value_counts(dropna=False)

Protocol
ParkTest           139
ValorPD            122
ValidationStudy     19
SuperPD_old         11
ClusterPD            7
SuperPD              5
InMotion             2
Name: count, dtype: int64

In [35]:
env_map = {
    'ParkTest':'Home-Global',
    'ValorPD': 'Clinic',
    'InMotion': 'PD Care Facility',
    'ValidationStudy': 'Clinic',
    'SuperPD': 'Clinic',
    'ClusterPD': 'Clinic',
    'SuperPD_old': 'Clinic',
    'SuperPD': 'Clinic',
    'RoutePD': 'PD Care Facility'
    
}

metadata['env'] = metadata['Protocol'].map(env_map)
metadata['env'].value_counts(dropna=False)

/tmp/ipykernel_2543543/1323468674.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata['env'] = metadata['Protocol'].map(env_map)


env
Clinic              164
Home-Global         139
PD Care Facility      2
Name: count, dtype: int64

In [36]:
# Define the preferred order
preferred_order = [
    "Home-Global",
    "PD Care Facility",
    "Clinic",
]

# Call the function with the preferred order
env_table_with_order = create_distribution_table_with_order(
    df=metadata,
    diagnosis_col="carrier",
    target_col="env",
    first_row_text="Recording Environment, n (%)",
    value_order=preferred_order
)
env_table_with_order

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,"Recording Environment, n (%)",Home-Global,0 (0.0%),139 (76.0%),139 (45.6%)
1,,PD Care Facility,0 (0.0%),2 (1.1%),2 (0.7%)
2,,Clinic,122 (100.0%),42 (23.0%),164 (53.8%)


In [37]:
total_table = pd.concat([
    pd_count_table, 
    sex_table_with_order, 
    age_table_with_order, 
    race_table_with_order, 
    env_table_with_order
], ignore_index=True)
total_table

,Demographic Property,Attribute,LRRK2 Carrier,Control,Total
0,,"Number of Participants, n (%)",122 (40.0%),183 (60.0%),305 (100%)
1,Sex,Female,71 (58.2%),105 (57.4%),176 (57.7%)
2,,Male,51 (41.8%),78 (42.6%),129 (42.3%)
3,"Age in years (range: 21.0 - 87.0, mean: 55.2),...",< 20,0 (0.0%),0 (0.0%),0 (0.0%)
4,,20 - 39,35 (28.7%),23 (12.6%),58 (19.0%)
5,,40 - 59,38 (31.1%),58 (31.7%),96 (31.5%)
6,,60 - 79,49 (40.2%),94 (51.4%),143 (46.9%)
7,,>= 80,0 (0.0%),1 (0.5%),1 (0.3%)
8,,Not Mentioned,0 (0.0%),7 (3.8%),7 (2.3%)
9,"Race, n (%)",White,116 (95.1%),177 (96.7%),293 (96.1%)


In [38]:
total_table.to_csv('../../data/demographic_table_demo_matched_cohort.csv', index=False)  